---
Accession Extraction Validation - Experiment Component

This module handles model inference on XML articles to extract accession numbers.
It supports multiple LLM providers with a unified interface.

Project: AI6129 Pathogen Tracking System
Version: 1.1

Model Configuration (Updated December 2025):
- Development: Claude Haiku 3.5
- Formal Experiments: Claude Sonnet 4 and Amazon Nova Pro via AWS Bedrock

Changes in v1.1:
- Added actual token capture from DSPy LM history
- Updated JSON loading to handle new validation_splits.json format
- Added cost calculation support
---

In [ ]:
!pip install dspy

In [ ]:
import json
import os
import re
import time
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from pathlib import Path
from typing import Dict, List, Set, Tuple, Optional, Any, Callable
from google.colab import drive
import xml.etree.ElementTree as ET

import dspy

In [ ]:
#CONFIG CONSTANTS

# DSPy model strings - validated for DSPy 3.x with LiteLLM
DSPY_MODEL_STRINGS = {
    # Baseline (Anthropic API - direct call, cost-effective)
    "claude-haiku-3.5": "anthropic/claude-3-5-haiku-20241022",
    
    # Formal Experiments (AWS Bedrock Singapore/APAC)
    # Claude Sonnet 4 - available in Singapore via APAC inference profile
    "claude-sonnet-4": "bedrock/apac.anthropic.claude-sonnet-4-20250514-v1:0",
    
    # Amazon Nova Pro - available in Singapore directly and via APAC profile
    "amazon-nova-pro": "bedrock/apac.amazon.nova-pro-v1:0",
}

# Alternative: Direct Singapore region model IDs (if APAC profile unavailable)
BEDROCK_SINGAPORE_DIRECT = {
    "amazon-nova-pro": "bedrock/amazon.nova-pro-v1:0",
    "amazon-nova-lite": "bedrock/amazon.nova-lite-v1:0",
    "amazon-nova-micro": "bedrock/amazon.nova-micro-v1:0",
}

# Default path for accession format rules file
DEFAULT_FORMAT_RULES_FILE = "accession_formats.md"

# Model pricing per 1M tokens (USD) - for cost calculation  # changed
MODEL_PRICING = {  # changed
    "claude-haiku-3.5": {"input": 0.80, "output": 4.00},  # changed
    "claude-sonnet-4": {"input": 3.00, "output": 15.00},  # changed
    "amazon-nova-pro": {"input": 0.80, "output": 3.20},  # changed
}  # changed

 ---
 TYPE DEFINITIONS
 ---

In [ ]:
class ModelProvider(Enum):
    """Supported LLM providers"""
    ANTHROPIC_API = "anthropic_api"
    GEMINI_API = "gemini_api"
    GOOGLE_COLAB = "google_colab"
    AWS_BEDROCK = "aws_bedrock"

class ExperimentStatus(Enum):
    """Status of document processing"""
    COMPLETED = "completed"
    FAILED = "failed"
    SKIPPED = "skipped"


@dataclass
class ModelConfig:
    """
    Configuration for a single model.

    Attributes:
        model_id: Short identifier (e.g., "gemini-2.0-flash")
        provider: Which LLM provider to use
        dspy_model_string: Full model string for DSPy
        max_tokens: Maximum output tokens
        temperature: Sampling temperature
        requests_per_minute: Rate limit
        region: AWS region for Bedrock (None for non-AWS)
    """
    model_id: str
    provider: ModelProvider
    dspy_model_string: str
    max_tokens: int = 4096
    temperature: float = 0.0
    requests_per_minute: int = 60
    region: Optional[str] = None


@dataclass
class ArticleContent:
    """Processed article ready for LLM input"""
    pmcid: str
    xml_filepath: str
    raw_xml: str
    extracted_text: str
    sections: Dict[str, str]
    tables: List[str]
    token_count_estimate: int


@dataclass
class ExtractionResult:
    """Result from processing a single document"""
    pmcid: str
    extracted_accessions: List[str]
    raw_model_output: str
    processing_time_ms: int
    token_count_input: int
    token_count_output: int
    tokens_are_estimated: bool  # changed - flag to indicate if tokens are estimates
    cost_usd: float  # changed - calculated cost for this inference
    model_id: str
    timestamp: str
    status: ExperimentStatus
    error_message: Optional[str]


@dataclass
class ExperimentCheckpoint:
    """Checkpoint for resuming interrupted experiments"""
    experiment_id: str
    model_id: str
    split_name: str
    completed_pmcids: Set[str]
    failed_pmcids: Dict[str, str]
    last_updated: str
    total_documents: int


@dataclass
class ExperimentMetadata:
    """Metadata about an experiment run"""
    experiment_id: str
    model_id: str
    dspy_version: str
    run_timestamp: str
    split_name: str
    total_documents: int
    completed_documents: int
    failed_documents: int
    total_processing_time_ms: int
    total_tokens_input: int  # changed - aggregate token tracking
    total_tokens_output: int  # changed
    total_cost_usd: float  # changed - aggregate cost


@dataclass
class ExperimentOutput:
    """Final output structure consumed by evaluation component"""
    experiment_metadata: ExperimentMetadata
    predictions: List[ExtractionResult]


@dataclass
class RateLimiter:
    """Sliding window rate limiter for API calls"""
    requests_per_minute: int
    request_timestamps: List[float] = field(default_factory=list)

---
 MODEL CONFIGURATION
---

In [ ]:
def create_model_configs() -> Dict[str, ModelConfig]:
    """
    Create configuration for all models to be tested.
    
    Returns:
        Mapping of model_id -> ModelConfig
    
    Models configured:
        1. claude-haiku-3.5: Baseline (Anthropic API, cost-effective)
        2. claude-sonnet-4: Comparison A (AWS Bedrock APAC)
        3. amazon-nova-pro: Comparison B (AWS Bedrock Singapore/APAC)
    """
    configs = {}

    # Baseline: Claude Haiku 3.5 via Anthropic API
    configs["claude-haiku-3.5"] = ModelConfig(
        model_id="claude-haiku-3.5",
        provider=ModelProvider.ANTHROPIC_API,
        dspy_model_string=DSPY_MODEL_STRINGS["claude-haiku-3.5"],
        max_tokens=4096,
        temperature=0.0,
        requests_per_minute=50,
        region=None
    )
    # Comparison A: Claude Sonnet 4 via AWS Bedrock APAC
    configs["claude-sonnet-4"] = ModelConfig(
        model_id="claude-sonnet-4",
        provider=ModelProvider.AWS_BEDROCK,
        dspy_model_string=DSPY_MODEL_STRINGS["claude-sonnet-4"],
        max_tokens=4096,
        temperature=0.0,
        requests_per_minute=50,
        region="ap-southeast-1"
    )
    
    # Comparison B: Amazon Nova Pro via AWS Bedrock APAC
    configs["amazon-nova-pro"] = ModelConfig(
        model_id="amazon-nova-pro",
        provider=ModelProvider.AWS_BEDROCK,
        dspy_model_string=DSPY_MODEL_STRINGS["amazon-nova-pro"],
        max_tokens=4096,
        temperature=0.0,
        requests_per_minute=60,
        region="ap-southeast-1"
    )
    
    return configs

In [ ]:
def initialise_dspy_model(config: ModelConfig) -> dspy.LM:
    """
    Initialise DSPy language model based on provider configuration.

    Parameters:
        config: Model configuration

    Returns:
        Configured DSPy LM object

    Notes:
        - Anthropic API requires ANTHROPIC_API_KEY environment variable
        - Gemini API requires GEMINI_API_KEY environment variable
        - Bedrock requires AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_REGION_NAME
    """
    lm = None

    if config.provider == ModelProvider.ANTHROPIC_API:
        # Anthropic API - requires ANTHROPIC_API_KEY env var
        lm = dspy.LM(
            config.dspy_model_string,
            temperature=config.temperature,
            max_tokens=config.max_tokens
        ) 
    elif config.provider == ModelProvider.GEMINI_API:
        # Gemini API - requires GEMINI_API_KEY env var
        lm = dspy.LM(
            config.dspy_model_string,
            temperature=config.temperature,
            max_tokens=config.max_tokens
        )

    elif config.provider == ModelProvider.AWS_BEDROCK:
        # AWS Bedrock - requires AWS credentials in environment
        # Set AWS_REGION_NAME to ap-southeast-1 for Singapore
        os.environ["AWS_REGION_NAME"] = config.region or "ap-southeast-1"
        
        lm = dspy.LM(
            config.dspy_model_string,
            temperature=config.temperature,
            max_tokens=config.max_tokens
        )
    
    return lm

---
 FORMAT RULES LOADING
---

In [ ]:
def load_format_rules(filepath: str) -> str:
    """
    Load accession number format rules from markdown file.
    
    The format rules provide context to the LLM about valid GenBank accession
    number formats, helping it distinguish real accessions from example patterns.
    
    Parameters:
        filepath: Path to markdown file containing format rules
    
    Returns:
        String content of format rules
    
    Raises:
        FileNotFoundError: If format rules file does not exist
        IOError: If file cannot be read
    """
    format_path = Path(filepath)

    if not format_path.exists():
        raise FileNotFoundError(f"Format rules file not found: {filepath}")
    
    try:
        with open(format_path, 'r', encoding='utf-8') as f:
            content = f.read()

        print(f"Format rules loaded from: {filepath} ({len(content)} characters)")
        return content
    
    except Exception as e:
        raise IOError(f"Error reading format rule file: {e}")

---
XML PROCESSING FUNCTIONS
---

In [ ]:
def load_xml_articles(xml_filepath: str) -> str:
    """Load raw XML content from file."""
    
    with open(xml_filepath, 'r', encoding='utf-8') as f:
        return f.read()
    
def extract_text_content(element: ET.Element) -> str:
    """Extract all text content from an XML element."""
    texts = []

    if element.text:
        texts.append(element.text.strip())

    for child in element:
        texts.append(extract_text_content(child))
        if child.tail:
            texts.append(child.tail.strip())    
    
    return " ".join(t for t in texts if t)

def extract_article_sections(raw_xml: str) -> Dict[str, str]:
    """
    Parse XML and extract content by section.

    Parameters:
        raw_xml: Raw XML content

    Returns:
        Mapping of section name -> section content
    """
    sections = {}

    try:
        root = ET.fromstring(raw_xml)
    except ET.ParseError:
        return sections
    
    #Extract abstract
    abstract_elem = root.find(".//abstract")
    if abstract_elem is not None:
        sections["abstract"] = extract_text_content(abstract_elem)

    #Extract body sections
    body_elem = root.find(".//body")
    if body_elem is not None:
        section_elem = body_elem.findall(".//sec")

        for section in section_elem:
            title_elem = section.find(".//title")
            section_title = title_elem.text if title_elem is not None and title_elem.text else ""
            section_content = extract_text_content(section)

            if section_title:
                sections[section_title.lower().strip()] = section_content
            else:
                sections[f"section_{len(sections)}"] = section_content
    
    #Extract back matter (data availability, acknowledgement)
    back_elem = root.find(".//back")
    if back_elem is not None:
        sections["back_matter"] = extract_text_content(back_elem)
    
    return sections

def extract_tables_from_xml(raw_xml: str) -> List[str]:
    """
    Extract table content from XML.

    Parameters:
        raw_xml: Raw XML content

    Returns:
        List of formatted table strings
    """
    tables = []

    try:
        root = ET.fromstring(raw_xml)
    except ET.ParseError:
        return tables
    
    table_wraps = root.findall(".//table-wrap")

    for table_wrap in table_wraps:
        label_elem = table_wrap.find(".//label")
        caption_elem = table_wrap.find(".//caption")

        label = label_elem.text if label_elem is not None and label_elem.text else ""
        caption = extract_text_content(caption_elem) if caption_elem is not None else ""

        table_content = convert_table_to_text(table_wrap)
        formatted_table = f"{label}: {caption}\n{table_content}"
        tables.append(formatted_table)
    
    return tables

In [ ]:
def convert_table_to_text(table_element: ET.Element) -> str:
    """
    Convert XML table element to readable text format.
    
    Format:
        Header1 | Header2 | Header3
        ----------------------------------------
        Value1  | Value2  | Value3
    """
    rows = []

    #process header 
    thead = table_element.find(".//thead")
    if thead is not None:
        header_row = thead.find(".//tr")
        if header_row is not None:
            cells = [extract_text_content(cell) for cell in header_row.findall(".//th")]
            rows.append("|".join(cells))
            rows.append("-" * 40)
    
    #Process body rows
    tbody = table_element.find(".//tbody")
    if tbody is not None:
        body_rows = tbody.findall(".//tr")
        for row in body_rows:
            cells = [extract_text_content(cell) for cell in row.findall(".//td")]
            rows.append("|".join(cells))
    
    return "\n".join(rows)

def estimate_token_count(text: str) -> int:
    """Estimate token count (approximately 4 characters per token)."""
    return len(text) // 4

def prepare_article_content(
    pmcid: str,
    xml_filepath: str,
    max_tokens: int = 100000
) -> ArticleContent:
    """
    Load and process XML article for LLM input.
    
    Parameters:
        pmcid: Document identifier
        xml_filepath: Path to XML file
        max_tokens: Maximum tokens for LLM context
    
    Returns:
        ArticleContent ready for extraction
    """
    raw_xml = load_xml_articles(xml_filepath)
    sections = extract_article_sections(raw_xml)
    tables = extract_tables_from_xml(raw_xml)

    extracted_text = compose_llm_input(sections, tables)
    token_estimate = estimate_token_count(extracted_text)

    if token_estimate > max_tokens:
        extracted_text = truncate_text_smart(extracted_text, max_tokens)
        token_estimate = max_tokens

    return ArticleContent(
        pmcid=pmcid,
        xml_filepath=xml_filepath,
        raw_xml=raw_xml,
        extracted_text=extracted_text,
        sections=sections,
        tables=tables,
        token_count_estimate=token_estimate
    )

def compose_llm_input(sections: Dict[str, str], tables: List[str]) -> str:
    """
    Compose sections and tables into LLM input text.
    
    Format:
        [SECTION: ABSTRACT]
        Abstract content here...
        
        [TABLES]
        [TABLE 1]
        Table content...
    """
    parts = []

    for section_name, section_content in sections.items():
        parts.append(f"[SECTION: {section_name.upper()}]")
        parts.append(section_content)
        parts.append("")
    
    if tables:
        parts.append("[TABLES]")
        for idx, table_content in enumerate(tables):
            parts.append(f"[TABLE {idx + 1}]")
            parts.append(table_content)
            parts.append("")
    
    return "\n".join(parts)


def truncate_text_smart(text: str, max_tokens: int) -> str:
    """
    Truncate text preserving beginning and end.
    
    Strategy:
        - Keep first 70% of allowed content (methods section)
        - Keep last 30% of allowed content (data availability)
        - Insert truncation marker in middle
    """

    max_chars = max_tokens * 4

    if len(text) <= max_chars:
        return text
    
    front_chars = int(max_chars * 0.7)
    back_chars = max_chars - front_chars - 50

    front_text = text[:front_chars]
    back_text = text[-back_chars:]

    return f"{front_text}\n\n[.... CONTENT TRUNCATED .....]\n\n{back_text}"

---
DSPY SIGNATURE AND MODULE
---

In [ ]:
class AccessionExtractionSignature(dspy.Signature):
    """
    Extract GenBank accession numbers from scientific article XML.
    
    Your task is to find all GenBank accession numbers mentioned in the article.
    
    GenBank accession numbers follow specific formats defined in the format_rules.
    Look for these accession numbers in:
    - Methods sections (sample/strain information)
    - Results sections (sequence data references)
    - Tables (strain collections, sequence identifiers)
    - Supplementary data references
    - Data availability statements
    
    CRITICAL EXTRACTION RULES:
    - Extract each accession ONLY ONCE (return unique accessions only, no duplicates)
    - Do NOT extract example accessions from the format_rules (e.g., AB123456, XM_123456)
    - Only extract accessions that actually appear in the article_text
    - The format_rules contain examples for reference - these are NOT real accessions
    - Do NOT extract journal article identifiers (e.g., s41588-021-00978-w)
    - Do NOT extract DOI components or reference numbers
    - GenBank accessions never contain hyphens (except RefSeq underscores)
    """

    format_rules: str = dspy.InputField(
        desc="Rules defining valid GenBank accession number formats. "
             "Contains format examples that should NOT be extracted as real accessions. "
             "Use these rules to identify and validate accession patterns."
    )
    
    article_text: str = dspy.InputField(
        desc="Full text of a scientific article from PubMed Central, "
             "containing sections such as abstract, methods, results, "
             "and data availability statements. May include tables."
    )
    
    accession_numbers: List[str] = dspy.OutputField(
        desc="List of UNIQUE database accession numbers found in the article. "
             "Only include accessions that appear in the xml_content."
             "If no accession numbers are present, return an empty list "
             "Do NOT include format examples from the rules, gene names, protein names, or other non-accession identifiers."
    )

class AccessionExtractor(dspy.Module):
    """
    DSPy module for accession extraction with chain-of-thought reasoning.
    
    Uses ChainOfThought to encourage step-by-step reasoning:
        1. Review format rules to understand valid accession patterns
        2. Identify sections likely to contain accessions
        3. Locate candidate accession patterns
        4. Validate against known formats
        5. Return final list of unique accessions
    """
    def __init__(self):
        super().__init__()
        self.predictor = dspy.ChainOfThought(AccessionExtractionSignature)
    
    def forward(self, format_rules: str, article_text: str) -> dspy.Prediction:
        """
        Extract accession numbers from article text using format rules as context.
        
        Parameters:
            format_rules: String containing accession format definitions
            article_text: Processed article text to extract from
        
        Returns:
            DSPy Prediction with accession_numbers and rationale
        """
        prediction = self.predictor(
            format_rules=format_rules,
            article_text=article_text
        )
        return prediction

def create_accession_extractor() -> AccessionExtractor:
    """Factory function to create the extractor module."""
    return AccessionExtractor()

---
RATE LIMITING AND RETRY
---

In [ ]:
def create_rate_limiter(requests_per_min: int) -> RateLimiter:
    """Create rate limiter for API calls."""    

    return RateLimiter(requests_per_minute=requests_per_min, request_timestamps=[])

def wait_for_rate_limit(rate_limiter: RateLimiter):
    """
    Block execution if rate limit would be exceeded.
    
    Algorithm:
        1. Remove timestamps older than 60 seconds
        2. If at limit, calculate wait time
        3. Sleep until oldest request exits window
        4. Record current request timestamp
    """
    current_time = time.time()
    window_start = current_time - 60.0

    #Remove timestamps outside window
    rate_limiter.request_timestamps = [
        ts for ts in rate_limiter.request_timestamps if ts > window_start
    ]

    requests_in_window = len(rate_limiter.request_timestamps)

    if requests_in_window >= rate_limiter.requests_per_minute:
        oldest_request = min(rate_limiter.request_timestamps)
        wait_time = (oldest_request + 60.0) - current_time + 0.1

        if wait_time > 0:
            print(f"Rate limit reached, waiting {wait_time:.1f} sec")
            time.sleep(wait_time)
    
    rate_limiter.request_timestamps.append(time.time())

def execute_with_retry(func: Callable, 
                       max_retries: int = 3,
                       base_delay: float = 1.0,
                       max_delay: float = 60.0
                       ) -> Any:
    """
    Execute function with exponential backoff retry.
    
    Parameters:
        func: Function to execute
        max_retries: Maximum retry attempts
        base_delay: Initial delay between retries (seconds)
        max_delay: Maximum delay cap (seconds)
    
    Returns:
        Function result if successful
    
    Raises:
        Exception: If all retries exhausted
    """
    delay = base_delay
    last_exception = None

    for attempt in range(max_retries):
        try:
            result = func()
            return result
        
        except Exception as e:
            last_exception = e
            error_name = type(e).__name__
            print(f"{error_name} on attempt {attempt + 1}, waiting {delay}s: {e}")
            time.sleep(delay)
            delay = min(delay * 2, max_delay)
    
    raise Exception(f"Failed after {max_retries} attempts: {last_exception}")

---
CHECKPOINTING
---

In [ ]:
def create_checkpoint_filepath(output_directory: str, experiment_id: str) -> str:
    """Generate filepath for checkpoint file."""
    return f"{output_directory}/checkpoint_{experiment_id}.json"

def load_checkpoint(checkpoint_filepath: str) -> Optional[ExperimentCheckpoint]:
    """Load existing checkpoint if present."""
    if not Path(checkpoint_filepath).exists():
        return None

    with open(checkpoint_filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

    return ExperimentCheckpoint(
        experiment_id=data['experiment_id'],
        model_id=data['model_id'],
        split_name=data['split_name'],
        completed_pmcids=set(data['completed_pmcids']),
        failed_pmcids=data['failed_pmcids'],
        last_updated=data['last_updated'],
        total_documents=data['total_documents']
    )

def save_checkpoint(checkpoint: ExperimentCheckpoint, checkpoint_filepath: str) -> None:
    """Save checkpoint to file."""
    checkpoint.last_updated = datetime.now().isoformat()

    data = {
        'experiment_id': checkpoint.experiment_id,
        'model_id': checkpoint.model_id,
        'split_name': checkpoint.split_name,
        'completed_pmcids': list(checkpoint.completed_pmcids),
        'failed_pmcids': checkpoint.failed_pmcids,
        'last_updated': checkpoint.last_updated,
        'total_documents': checkpoint.total_documents
    }

    with open(checkpoint_filepath, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2)

def create_new_checkpoint(experiment_id: str,
                          model_id: str,
                          split_name: str,
                          total_documents: int
                          ) -> ExperimentCheckpoint:
    """Create a fresh checkpoint."""
    return ExperimentCheckpoint(
        experiment_id=experiment_id,
        model_id=model_id,
        split_name=split_name,
        completed_pmcids=set(),
        failed_pmcids={},
        last_updated=datetime.now().isoformat(),
        total_documents=total_documents
    )

---
TOKEN AND COST TRACKING (NEW IN v1.1)
---

In [ ]:
def get_actual_token_usage_from_dspy() -> Tuple[int, int, bool]:  # changed - new function
    """  # changed
    Extract actual token usage from DSPy LM history.  # changed
      # changed
    DSPy stores API response metadata in the LM's history attribute.  # changed
    This function attempts to retrieve actual token counts from the last API call.  # changed
      # changed
    Returns:  # changed
        Tuple of (input_tokens, output_tokens, is_actual)  # changed
        - input_tokens: Number of prompt/input tokens  # changed
        - output_tokens: Number of completion/output tokens  # changed
        - is_actual: True if values are from API, False if unavailable  # changed
    """  # changed
    try:  # changed
        lm = dspy.settings.lm  # changed
        
        # DSPy stores call history in lm.history  # changed
        if hasattr(lm, 'history') and lm.history:  # changed
            last_call = lm.history[-1]  # changed
            
            # Check for usage info in different possible locations  # changed
            usage = None  # changed
            
            # Try direct usage field  # changed
            if isinstance(last_call, dict) and 'usage' in last_call:  # changed
                usage = last_call['usage']  # changed
            # Try response.usage (some providers)  # changed
            elif hasattr(last_call, 'usage'):  # changed
                usage = last_call.usage  # changed
            # Try nested in response dict  # changed
            elif isinstance(last_call, dict) and 'response' in last_call:  # changed
                response = last_call['response']  # changed
                if isinstance(response, dict) and 'usage' in response:  # changed
                    usage = response['usage']  # changed
            
            if usage:  # changed
                # Handle dict-style usage  # changed
                if isinstance(usage, dict):  # changed
                    input_tokens = usage.get('prompt_tokens', 0) or usage.get('input_tokens', 0)  # changed
                    output_tokens = usage.get('completion_tokens', 0) or usage.get('output_tokens', 0)  # changed
                # Handle object-style usage  # changed
                else:  # changed
                    input_tokens = getattr(usage, 'prompt_tokens', 0) or getattr(usage, 'input_tokens', 0)  # changed
                    output_tokens = getattr(usage, 'completion_tokens', 0) or getattr(usage, 'output_tokens', 0)  # changed
                
                if input_tokens > 0 or output_tokens > 0:  # changed
                    return input_tokens, output_tokens, True  # changed
        
        return 0, 0, False  # changed
    
    except Exception as e:  # changed
        print(f"Warning: Could not retrieve token usage from DSPy: {e}")  # changed
        return 0, 0, False  # changed


def calculate_cost(model_id: str, input_tokens: int, output_tokens: int) -> float:  # changed - new function
    """  # changed
    Calculate cost in USD for a single inference.  # changed
      # changed
    Parameters:  # changed
        model_id: Model identifier (must match keys in MODEL_PRICING)  # changed
        input_tokens: Number of input tokens  # changed
        output_tokens: Number of output tokens  # changed
      # changed
    Returns:  # changed
        Cost in USD  # changed
    """  # changed
    pricing = MODEL_PRICING.get(model_id, {"input": 0, "output": 0})  # changed
    input_cost = (input_tokens / 1_000_000) * pricing["input"]  # changed
    output_cost = (output_tokens / 1_000_000) * pricing["output"]  # changed
    return input_cost + output_cost  # changed

---
OUTPUT PARSING
---

In [ ]:
def parse_model_output(prediction: dspy.Prediction) -> List[str]:
    """
    Parse DSPy prediction to extract accession list.
    
    Parameters:
        prediction: DSPy Prediction object
    
    Returns:
        List of accession strings
    """
    accessions_raw = getattr(prediction, 'accession_numbers', [])

    if accessions_raw is None:
        return []

    if isinstance(accessions_raw, str):
        accessions_raw = re.split(r'[,\n]', accessions_raw)
    
    accessions = []
    for acc in accessions_raw:
        if acc is None:
            continue
        cleaned = str(acc).strip()
        if cleaned:
            accessions.append(cleaned)
    
    return accessions

def create_extraction_result(pmcid: str,
                             accessions: List[str],
                             raw_output: str,
                             processing_time_ms: int,
                             token_count_input: int,
                             token_count_output: int,
                             tokens_are_estimated: bool,  # changed - new parameter
                             cost_usd: float,  # changed - new parameter
                             model_id: str
                             ) -> ExtractionResult:
    """Create successful extraction result."""
    return ExtractionResult(
        pmcid=pmcid,
        extracted_accessions=accessions,
        raw_model_output=raw_output,
        processing_time_ms=processing_time_ms,
        token_count_input=token_count_input,
        token_count_output=token_count_output,
        tokens_are_estimated=tokens_are_estimated,  # changed
        cost_usd=cost_usd,  # changed
        model_id=model_id,
        timestamp=datetime.now().isoformat(),
        status=ExperimentStatus.COMPLETED,
        error_message=None
    )

def create_failed_result(pmcid: str,
                         model_id: str,
                         error_message: str
                         ) -> ExtractionResult:
    """Create failed extraction result."""
    return ExtractionResult(
        pmcid=pmcid,
        extracted_accessions=[],
        raw_model_output="",
        processing_time_ms=0,
        token_count_input=0,
        token_count_output=0,
        tokens_are_estimated=True,  # changed
        cost_usd=0.0,  # changed
        model_id=model_id,
        timestamp=datetime.now().isoformat(),
        status=ExperimentStatus.FAILED,
        error_message=error_message
    )

---
MAIN PROCESSING FUNCTIONS
---

In [ ]:
def process_single_document(pmcid: str,
                            xml_filepath: str,
                            extractor: AccessionExtractor,
                            format_rules: str,
                            model_id: str,
                            rate_limiter: RateLimiter
                            ) -> ExtractionResult:
    """
    Process a single document to extract accessions.
    
    Parameters:
        pmcid: Document identifier
        xml_filepath: Path to XML file
        extractor: AccessionExtractor module
        format_rules: Accession format rules as context
        model_id: Model identifier
        rate_limiter: Rate limiter for API calls
    
    Returns:
        ExtractionResult with predictions
    """
    try:
        # Prepare article content
        article = prepare_article_content(pmcid, xml_filepath)

        # Rate limit
        wait_for_rate_limit(rate_limiter)

        # Run extraction with timing
        start_time = time.time()

        def run_extraction():
            return extractor(
                format_rules,
                article.extracted_text
            )
        
        prediction = execute_with_retry(run_extraction)

        end_time = time.time()
        processing_time_ms = int((end_time - start_time) * 1000)

        # Parse output
        accessions = parse_model_output(prediction)

        # Get raw output for debugging
        raw_output = str(getattr(prediction, 'accession_numbers', ''))
        if hasattr(prediction, 'rationale'):
            raw_output = f"Rationale: {prediction.rationale}\n\nAccessions: {raw_output}"
        
        # Get actual token counts from DSPy if available  # changed
        actual_input, actual_output, is_actual = get_actual_token_usage_from_dspy()  # changed
        
        if is_actual:  # changed
            token_count_input = actual_input  # changed
            token_count_output = actual_output  # changed
            tokens_are_estimated = False  # changed
        else:  # changed
            # Fallback to estimates  # changed
            token_count_input = article.token_count_estimate  # changed
            token_count_output = estimate_token_count(raw_output)  # changed
            tokens_are_estimated = True  # changed
        
        # Calculate cost  # changed
        cost_usd = calculate_cost(model_id, token_count_input, token_count_output)  # changed

        return create_extraction_result(
            pmcid=pmcid,
            accessions=accessions,
            raw_output=raw_output,
            processing_time_ms=processing_time_ms,
            token_count_input=token_count_input,
            token_count_output=token_count_output,
            tokens_are_estimated=tokens_are_estimated,  # changed
            cost_usd=cost_usd,  # changed
            model_id=model_id
        )

    except Exception as e:
        return create_failed_result(
            pmcid=pmcid,
            model_id=model_id,
            error_message=str(e)
        )

def load_pmcid_to_xml_mapping(xml_directory: str,
                              pmcid_list: List[str]
                              ) -> Dict[str, str]:
    """
    Create mapping of PMCID to XML file paths.
    
    Parameters:
        xml_directory: Directory containing XML files
        pmcid_list: List of PMCIDs to find
    
    Returns:
        Mapping of PMCID -> filepath
    """
    xml_dir = Path(xml_directory)
    mapping = {}

    pmcid_set = set(pmcid_list)
    
    for xml_file in xml_dir.glob("*.xml"):
        # Extract PMCID from filename
        filename = xml_file.stem
        pmcid = filename.split('_')[0]

        if pmcid in pmcid_set:
            mapping[pmcid] = str(xml_file)
    
    return mapping


def load_split_pmcids(ground_truth_filepath: str, split_name: str) -> List[str]:  # changed - new helper function
    """  # changed
    Load PMCIDs for a specific split from validation_splits.json.  # changed
      # changed
    Handles both old format (list) and new format (dict with 'files' key).  # changed
      # changed
    Parameters:  # changed
        ground_truth_filepath: Path to validation_splits.json  # changed
        split_name: Name of split (development/validation/test)  # changed
      # changed
    Returns:  # changed
        List of PMCIDs in the split  # changed
    """  # changed
    with open(ground_truth_filepath, 'r', encoding='utf-8') as f:  # changed
        gt_data = json.load(f)  # changed
      # changed
    split_data = gt_data['splits'].get(split_name, {})  # changed
      # changed
    # Handle both old format (list) and new format (dict with 'files' key)  # changed
    if isinstance(split_data, dict):  # changed
        pmcid_list = split_data.get('files', [])  # changed
    else:  # changed
        # Old format: split_data is already a list  # changed
        pmcid_list = split_data  # changed
      # changed
    return pmcid_list  # changed


def run_experiment(model_config: ModelConfig,
                   split_name: str,
                   ground_truth_filepath: str,
                   xml_directory: str,
                   format_rules_filepath: str,
                   output_directory: str,
                   resume_from_checkpoint: bool = True
                  ) -> ExperimentOutput:
    """
    Run experiment for a single model.
    
    Parameters:
        model_config: Model configuration
        split_name: Data split to process
        ground_truth_filepath: Path to validation_splits.json
        xml_directory: Directory with XML files
        format_rules_filepath: Path to accession_formats.md
        output_directory: Directory for outputs
        resume_from_checkpoint: Whether to resume if checkpoint exists
    
    Returns:
        ExperimentOutput with all predictions
    """
    # Generate experiment ID
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    experiment_id = f"{model_config.model_id}_{split_name}_{timestamp_str}"

    # Create output directory
    output_dir = Path(output_directory)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Load accession format rules
    print(f"Loading format rules from: {format_rules_filepath}")
    format_rules = load_format_rules(format_rules_filepath)

    # Load split pmcids using the new helper function  # changed
    pmcid_list = load_split_pmcids(ground_truth_filepath, split_name)  # changed
    total_documents = len(pmcid_list)

    print(f"Experiment: {experiment_id}")
    print(f"Model: {model_config.model_id}")
    print(f"Split: {split_name} ({total_documents} documents)")

    # Load or create checkpoint
    checkpoint_filepath = create_checkpoint_filepath(output_directory, experiment_id)
    checkpoint = None

    if resume_from_checkpoint:
        checkpoint = load_checkpoint(checkpoint_filepath)

    if checkpoint is None:
        checkpoint = create_new_checkpoint(experiment_id, model_config.model_id,
                                           split_name, total_documents
                                           )
    
    # Initialise dspy
    lm = initialise_dspy_model(model_config)
    dspy.configure(lm=lm)

    # Create extractor and rate limiter
    extractor = create_accession_extractor()
    rate_limiter = create_rate_limiter(model_config.requests_per_minute)

    # Load pmcid to XML mapping
    pmcid_to_xml = load_pmcid_to_xml_mapping(xml_directory, pmcid_list)

    # Process documents
    predictions = []
    total_processing_time_ms = 0
    total_tokens_input = 0  # changed - aggregate tracking
    total_tokens_output = 0  # changed
    total_cost_usd = 0.0  # changed

    for idx, pmcid in enumerate(pmcid_list):
        # Skip if completed
        if pmcid in checkpoint.completed_pmcids:
            print(f"  [{idx+1}/{total_documents}] {pmcid} - SKIPPED (already completed)")
            continue
    
        # Check if XML exists
        if pmcid not in pmcid_to_xml:
            result = create_failed_result(pmcid, model_config.model_id, "XML file not found")
            checkpoint.failed_pmcids[pmcid] = "XML file not found"
        else:
            print(f"  [{idx+1}/{total_documents}] {pmcid} - Processing...")
            result = process_single_document(
                pmcid=pmcid,
                xml_filepath=pmcid_to_xml[pmcid],
                extractor=extractor,
                format_rules=format_rules,
                model_id=model_config.model_id,
                rate_limiter=rate_limiter
            )
            
            if result.status == ExperimentStatus.COMPLETED:
                checkpoint.completed_pmcids.add(pmcid)
                token_info = "actual" if not result.tokens_are_estimated else "est"  # changed
                print(f"    Found {len(result.extracted_accessions)} accessions "
                      f"(tokens: {result.token_count_input}/{result.token_count_output} [{token_info}], "
                      f"cost: ${result.cost_usd:.4f})")  # changed
            else:
                checkpoint.failed_pmcids[pmcid] = result.error_message
                print(f"   FAILED: {result.error_message}")

        predictions.append(result)
        total_processing_time_ms += result.processing_time_ms
        total_tokens_input += result.token_count_input  # changed
        total_tokens_output += result.token_count_output  # changed
        total_cost_usd += result.cost_usd  # changed

        # Save checkpoint periodically
        if (idx + 1) % 10 == 0:
            save_checkpoint(checkpoint, checkpoint_filepath)
            print(f"    [Checkpoint saved. Running totals: {total_tokens_input} in / "
                  f"{total_tokens_output} out tokens, ${total_cost_usd:.4f}]")  # changed
    
    # Final checkpoint save
    save_checkpoint(checkpoint, checkpoint_filepath)

    # Create output
    metadata = ExperimentMetadata(
        experiment_id=experiment_id,
        model_id=model_config.model_id,
        dspy_version=dspy.__version__ if hasattr(dspy, '__version__') else "unknown",
        run_timestamp=datetime.now().isoformat(),
        split_name=split_name,
        total_documents=total_documents,
        completed_documents=len(checkpoint.completed_pmcids),
        failed_documents=len(checkpoint.failed_pmcids),
        total_processing_time_ms=total_processing_time_ms,
        total_tokens_input=total_tokens_input,  # changed
        total_tokens_output=total_tokens_output,  # changed
        total_cost_usd=total_cost_usd  # changed
    )

    output = ExperimentOutput(
        experiment_metadata=metadata,
        predictions=predictions
    )

    # Save output
    output_filepath = output_dir / f"{model_config.model_id}_{split_name}_predictions.json"
    save_experiment_output(output, str(output_filepath))

    print(f"\nExperiment complete: {output_filepath}")
    print(f"  Completed: {metadata.completed_documents}")
    print(f"  Failed: {metadata.failed_documents}")
    print(f"  Total tokens: {total_tokens_input} input / {total_tokens_output} output")  # changed
    print(f"  Total cost: ${total_cost_usd:.4f}")  # changed

    return output
             

def save_experiment_output(output: ExperimentOutput, output_filepath: str) -> None:
    """Save experiment output to JSON file."""
    predictions_list = []

    for result in output.predictions:
        predictions_list.append({
            "pmcid": result.pmcid,
            "extracted_accessions": result.extracted_accessions,
            "raw_model_output": result.raw_model_output,
            "processing_time_ms": result.processing_time_ms,
            "token_count_input": result.token_count_input,
            "token_count_output": result.token_count_output,
            "tokens_are_estimated": result.tokens_are_estimated,  # changed
            "cost_usd": result.cost_usd,  # changed
            "timestamp": result.timestamp,
            "status": result.status.value,
            "error_message": result.error_message
        })
    
    output_dict = {
        "experiment_metadata": {
            "experiment_id": output.experiment_metadata.experiment_id,
            "model_id": output.experiment_metadata.model_id,
            "dspy_version": output.experiment_metadata.dspy_version,
            "run_timestamp": output.experiment_metadata.run_timestamp,
            "split_name": output.experiment_metadata.split_name,
            "total_documents": output.experiment_metadata.total_documents,
            "completed_documents": output.experiment_metadata.completed_documents,
            "failed_documents": output.experiment_metadata.failed_documents,
            "total_processing_time_ms": output.experiment_metadata.total_processing_time_ms,
            "total_tokens_input": output.experiment_metadata.total_tokens_input,  # changed
            "total_tokens_output": output.experiment_metadata.total_tokens_output,  # changed
            "total_cost_usd": output.experiment_metadata.total_cost_usd  # changed
        },
        "predictions": predictions_list
    }

    with open(output_filepath, 'w', encoding='utf-8') as f:
        json.dump(output_dict, f, indent=2)

def run_all_experiments(model_ids: List[str],
                        split_name: str,
                        ground_truth_filepath: str,
                        xml_directory: str,
                        format_rules_filepath: str,
                        output_directory: str
                        ) -> List[ExperimentOutput]:    
    """
    Run experiments for multiple models.
    
    Parameters:
        model_ids: List of model IDs to test
        split_name: Data split to process
        ground_truth_filepath: Path to validation_splits.json
        xml_directory: Directory with XML files
        format_rules_filepath: Path to accession_formats.md
        output_directory: Directory for outputs
    
    Returns:
        List of ExperimentOutput for each model
    """

    configs = create_model_configs()
    outputs = []

    for model_id in model_ids:
        if model_id not in configs:
            print(f"Unknown model: {model_id}")
            continue
        
        config = configs[model_id]

        try: 
            output = run_experiment(
                model_config=config,
                split_name=split_name,
                ground_truth_filepath=ground_truth_filepath,
                xml_directory=xml_directory,
                format_rules_filepath=format_rules_filepath,
                output_directory=output_directory,
                resume_from_checkpoint=True
            )
            outputs.append(output)
        
        except Exception as e:
            print(f"Experiment failed for {model_id}: {e}")
    
    print(f"\nAll experiments complete. {len(outputs)} successful.")
    return outputs

In [ ]:
# Set path directory
BASE_DIR = Path("/content/drive/MyDrive/AI6129")
OUTPUT_DIR = BASE_DIR / "accession/experiment_outputs"
GROUND_TRUTH_FILE = BASE_DIR / "accession/validation_splits/validation_splits.json"
FORMAT_RULES = BASE_DIR / "accession_formats.md"
XML_FILES_DIR = BASE_DIR / "xml"

# Mount google drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
# Run experiment for development set with Haiku 3.5 baseline
# Set your Anthropic API key first:
# os.environ["ANTHROPIC_API_KEY"] = "your-api-key-here"

output = run_experiment(
    model_config=create_model_configs()["claude-haiku-3.5"],
    split_name="development",
    ground_truth_filepath=str(GROUND_TRUTH_FILE),
    xml_directory=str(XML_FILES_DIR),
    format_rules_filepath=str(FORMAT_RULES),
    output_directory=str(OUTPUT_DIR),
    resume_from_checkpoint=True
)